In [ ]:
# initialize dependencies
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
from potential_grad_surrogate import GatedPotentialSurrogate
from sklearn.preprocessing import RobustScaler

In [ ]:
# Load preprocessed data (saved by analyze_data.ipynb after feature pruning)
DATA_CSV = '../data/datasets/processed_data.csv'
data = pd.read_csv(DATA_CSV)

target_col = 'target_vol'
lagged_vol_feature = 'lagged_vol'
feature_columns = [col for col in data.columns if col != target_col]
if lagged_vol_feature not in feature_columns:
    raise ValueError('Lagged volatility feature missing after preprocessing.')

print(f"Loaded {len(data)} rows, {len(feature_columns)} features from {DATA_CSV}")


In [ ]:
"""Define Variables"""
dt = 1  # 1 discrete daily step
batch_size = 128
K = 3  # number of expert potential surfaces

# for training loop 
lambda_variational = 0.05   # variational: penalize steep gradients -> less overprediction and interpretable
lambda_directional = 0.00
lambda_huber = 0.0          # Huber loss: balanced attention to peaks AND dips
lambda_bias = 0.0           # penalizes systematic over/underprediction
lambda_entropy = 0.05       # reduced: allow gate to specialize across regimes
num_epochs = 300

# for validation loop
criterion = nn.MSELoss()

In [ ]:
features = feature_columns.copy()

# Define fast vs slow features
# Fast: price-derived daily features
# Slow: macro/rates features
fast_feature_names = ['close', 'volume', 'returns', 'lagged_vol']
slow_feature_names = [f for f in features if f not in fast_feature_names]
slow_indices = [features.index(f) for f in slow_feature_names]
slow_dim = len(slow_feature_names)
print(f"Fast features ({len(fast_feature_names)}): {fast_feature_names}")
print(f"Slow features ({slow_dim}): {slow_feature_names}")

# split and copy to avoid SettingWithCopyWarning
train_split = int(0.8 * len(data))
train_data = data.iloc[:train_split].copy()
test_data = data.iloc[train_split:].copy()

train_y = train_data.pop(target_col)
test_y = test_data.pop(target_col)

vol_index = features.index(lagged_vol_feature)

# RobustScaler uses median/IQR - much less sensitive to the fat tails
scaler = RobustScaler()
train_X = scaler.fit_transform(train_data[features])
test_X = scaler.transform(test_data[features])

scaled_train_df = pd.DataFrame(train_X, columns=features)
scaled_test_df = pd.DataFrame(test_X, columns=features)

vol_center = scaler.center_[vol_index]
vol_scale = scaler.scale_[vol_index]

# Target delta in scaled vol space
train_lagged_vol = train_data[lagged_vol_feature].values
test_lagged_vol = test_data[lagged_vol_feature].values

train_y_delta = (train_y.values - train_lagged_vol) / vol_scale
test_y_delta = (test_y.values - test_lagged_vol) / vol_scale

tensor_train_X = torch.tensor(train_X, dtype=torch.float32)
tensor_train_y = torch.tensor(train_y_delta, dtype=torch.float32)
tensor_test_X = torch.tensor(test_X, dtype=torch.float32)
tensor_test_y = torch.tensor(test_y_delta, dtype=torch.float32)

# Extract slow features as separate tensors for the gate
tensor_train_slow = tensor_train_X[:, slow_indices]
tensor_test_slow = tensor_test_X[:, slow_indices]

train_dataset = TensorDataset(tensor_train_X, tensor_train_slow, tensor_train_y)
test_dataset = TensorDataset(tensor_test_X, tensor_test_slow, tensor_test_y)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
model = GatedPotentialSurrogate(
    input_dim=len(features),
    slow_dim=slow_dim,
    K=K,
    hidden_layers=3,
    neurons_per_layer=256,
    activation_function=nn.ReLU
)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
huber = nn.HuberLoss(delta=1.0)

# Early stopping
best_val_loss = float('inf')
patience = 500
patience_counter = 0
best_state = None

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for batch_X, batch_slow, batch_y in train_loader:
        optimizer.zero_grad()
        batch_X = batch_X.clone().detach().requires_grad_(True)

        potentials, log_var, gate_weights, expert_potentials = model(batch_X, batch_slow)
        var = torch.exp(log_var) + 1e-10

        batch_gradients = torch.autograd.grad(
            potentials,
            batch_X,
            grad_outputs=torch.ones_like(potentials),
            create_graph=True
        )[0]

        batch_predictions = -batch_gradients[:, vol_index] * dt
        batch_direction = torch.sign(batch_predictions) * torch.sign(batch_y)

        gaussian_nll = (0.5 * torch.log(var) + 0.5 * (batch_y - batch_predictions) ** 2 / var).mean()
        huber_loss = huber(batch_predictions, batch_y)
        bias_penalty = (batch_predictions.mean() - batch_y.mean()) ** 2
        variational_loss = (batch_gradients ** 2).mean()
        directional_loss = (1 - batch_direction).mean()

        gate_entropy = -(gate_weights * torch.log(gate_weights + 1e-10)).sum(dim=1).mean()
        entropy_penalty = -gate_entropy

        loss = (gaussian_nll
                + lambda_huber * huber_loss
                + lambda_bias * bias_penalty
                + lambda_variational * variational_loss
                + lambda_directional * directional_loss
                + lambda_entropy * entropy_penalty)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()

    scheduler.step()

    # Validation loss for early stopping (MSE on delta predictions)
    if (epoch + 1) % 50 == 0:
        model.eval()
        val_mse = 0.0
        for vX, vS, vY in test_loader:
            vX = vX.clone().detach().requires_grad_(True)
            vP, _, _, _ = model(vX, vS)
            vG = torch.autograd.grad(vP, vX, grad_outputs=torch.ones_like(vP))[0]
            vPred = -vG[:, vol_index] * dt
            val_mse += ((vPred - vY) ** 2).mean().item()
        val_mse /= len(test_loader)

        if val_mse < best_val_loss:
            best_val_loss = val_mse
            patience_counter = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 0

        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}, best val MSE: {best_val_loss:.6f}')
            break

    if (epoch + 1) % 100 == 0:
        with torch.no_grad():
            sample_gate = model.gate(tensor_train_slow[:256])
            gate_mean = sample_gate.mean(dim=0).numpy()
        gate_str = ', '.join(f'{g:.3f}' for g in gate_mean)
        lr_now = scheduler.get_last_lr()[0]
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(train_loader):.4f}, ValMSE: {val_mse:.6f}, LR: {lr_now:.2e}, Gate avg: [{gate_str}]')

# Restore best model
if best_state is not None:
    model.load_state_dict(best_state)
    print(f'Restored best model (val MSE: {best_val_loss:.6f})')

In [ ]:
model.eval()
model_vol_preds = []
model_vol_mean_preds = []
model_log_vars = []
all_gate_weights = []
test_loss = 0.0
count_batches = 0

for batch_X, batch_slow, batch_y in test_loader:
    count_batches += 1
    batch_X = batch_X.clone().detach().requires_grad_(True)

    potentials, log_var, gate_weights, _ = model(batch_X, batch_slow)
    var = torch.exp(log_var) + 1e-8

    grads = torch.autograd.grad(
        potentials,
        batch_X,
        grad_outputs=torch.ones_like(potentials),
        create_graph=False
    )[0]

    delta_mean = -grads[:, vol_index] * dt
    test_loss += criterion(delta_mean, batch_y).item()

    lagged_vol_raw = batch_X[:, vol_index].detach() * vol_scale + vol_center
    pred_vol_mean_raw = lagged_vol_raw + delta_mean.detach() * vol_scale
    pred_vol_raw = pred_vol_mean_raw + torch.randn_like(delta_mean) * torch.sqrt(2 * var * dt).detach() * vol_scale

    model_vol_mean_preds.append(pred_vol_mean_raw.cpu())
    model_vol_preds.append(pred_vol_raw.cpu())
    model_log_vars.append(log_var.detach().cpu())
    all_gate_weights.append(gate_weights.detach().cpu())

print(f"Test Loss (mean delta forecast): {test_loss / len(test_loader)}")

model_vol_mean_preds = torch.cat(model_vol_mean_preds).numpy()
model_vol_preds = torch.cat(model_vol_preds).numpy()
model_log_vars = torch.cat(model_log_vars).numpy()
all_gate_weights = torch.cat(all_gate_weights).numpy()

# Show gate usage across test set
print(f"\nGate weight distribution across test set (mean per expert):")
for k in range(K):
    print(f"  Expert {k}: {all_gate_weights[:, k].mean():.4f} (std {all_gate_weights[:, k].std():.4f})")

In [ ]:
import numpy as np
from arch import arch_model

def estimate_garch_vol(returns: np.ndarray) -> float:
    """
    Fit GARCH(1,1) on historical returns, return 1-day ahead vol forecast (same units as returns).
    """
    # arch expects returns in %, scale if your returns are in decimal
    model = arch_model(returns * 100, vol="Garch", p=1, q=1, rescale=False)
    res = model.fit(disp="off")
    
    # 1-step ahead forecast
    forecast = res.forecast(horizon=1)
    var_1d = forecast.variance.iloc[-1, 0]
    
    return np.sqrt(var_1d) / 100  # back to decimal

In [ ]:
# arch predictions for test set
arch_vol_preds = []
returns_series = data['returns'].values
for i in range(len(test_y)):
    hist_returns = returns_series[:train_split + i]
    hist_returns = hist_returns[~np.isnan(hist_returns)]
    vol_pred = estimate_garch_vol(hist_returns)
    arch_vol_preds.append(vol_pred)

realized_vols = test_y.values  # now EWMA vol (smooth)

# plot model_preds, arch_preds, and realized vols
import matplotlib.pyplot as plt
N = len(test_y)
plt.figure(figsize=(12, 6))
plt.plot(realized_vols[-N:], label='Realized EWMA Vol')
plt.plot(model_vol_mean_preds[-N:], label='Surrogate Model Vol')
# GARCH is directly comparable to EWMA (now that EWMA is baseline)
plt.plot(np.array(arch_vol_preds[-N:]), label='GARCH(1,1) Vol')
plt.legend()

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

vix_feature = 'vix_close'
vix_index = features.index(vix_feature)
vix_center = scaler.center_[vix_index]
vix_iqr = scaler.scale_[vix_index]

# Conditioning feature used to bucket regimes
spread_feature = 'yield_curve_yield_spread_10y_2y'
spread_index = features.index(spread_feature)
spread_center = scaler.center_[spread_index]
spread_iqr = scaler.scale_[spread_index]

vol_grid = np.linspace(0.001, 0.125, 50)
vix_grid = np.linspace(scaled_test_df[vix_feature].min() * vix_iqr + vix_center,
                       scaled_test_df[vix_feature].max() * vix_iqr + vix_center, 50)
X_mesh, Y_mesh = np.meshgrid(vol_grid, vix_grid)

# -- Build regime templates from historical data --
# Bucket training observations by yield spread percentile, then take the median of ALL features within each bucket
# This gives a realistic feature vector where every macro/market variable is correlated with the yield-curve regime
spread_raw_train = scaled_train_df[spread_feature].values * spread_iqr + spread_center
p33, p66 = np.percentile(spread_raw_train, [33, 66])

bucket_labels = np.where(spread_raw_train < p33, 'low',
                np.where(spread_raw_train < p66, 'mid', 'high'))

regime_defs = {
    'Inverted / Tight Spread (P0‑P33)': 'low',
    'Normal Spread (P33‑P66)': 'mid',
    'Steep / Wide Spread (P66‑P100)': 'high',
}

# For each bucket, compute the median feature vector across ALL features
regime_templates = {}
for label, bucket in regime_defs.items():
    mask = bucket_labels == bucket
    median_vec = scaled_train_df.iloc[mask].median().values  # all features at once
    regime_templates[label] = torch.tensor(median_vec, dtype=torch.float32).unsqueeze(0)

def eval_surface(mesh_X, mesh_Y, forward_fn, base_template):
    """Evaluate a surface using base_template as the starting feature vector."""
    Z = np.zeros_like(mesh_X)
    for i in range(mesh_X.shape[0]):
        for j in range(mesh_X.shape[1]):
            inp = base_template.clone()
            inp[0, vol_index] = (mesh_X[i, j] - vol_center) / vol_scale
            inp[0, vix_index] = (mesh_Y[i, j] - vix_center) / vix_iqr
            inp = inp.detach().requires_grad_(True)
            pot = forward_fn(inp)
            Z[i, j] = pot.item()
    return Z

n_regimes = len(regime_defs)
n_cols = K + 1  # experts + gated
fig, axes = plt.subplots(n_regimes, n_cols, figsize=(6 * n_cols, 5.5 * n_regimes),
                         subplot_kw={'projection': '3d'})
cmaps = ['Reds', 'Blues', 'Greens', 'Oranges', 'Purples']

for row, (regime_label, regime_base) in enumerate(regime_templates.items()):
    regime_slow = regime_base[:, slow_indices]

    # Gate weights under this regime
    with torch.no_grad():
        gate_here = model.gate(regime_slow).squeeze().numpy()

    # Each expert column
    for k in range(K):
        Z_k = eval_surface(X_mesh, Y_mesh,
            lambda inp, k=k: model.forward_expert(inp, k)[0], regime_base)
        ax = axes[row, k]
        ax.plot_surface(X_mesh, Y_mesh, Z_k, alpha=0.8, cmap=cmaps[k % len(cmaps)])
        ax.set_xlabel('Lagged Vol', fontsize=8)
        ax.set_ylabel('VIX', fontsize=8)
        ax.set_zlabel('Potential', fontsize=8)
        ax.view_init(elev=30, azim=240)
        ax.set_facecolor('white')
        for pane in [ax.xaxis, ax.yaxis, ax.zaxis]:
            pane.set_pane_color((1, 1, 1, 0))
        if row == 0:
            ax.set_title(f'Expert {k}', fontsize=11, fontweight='bold')
        ax.text2D(0.02, 0.92, f'w={gate_here[k]:.3f}', transform=ax.transAxes, fontsize=9)

    # Gated combination column
    Z_gated = eval_surface(X_mesh, Y_mesh,
        lambda inp, _slow=regime_slow: model(inp, _slow)[0], regime_base)
    ax = axes[row, -1]
    ax.plot_surface(X_mesh, Y_mesh, Z_gated, alpha=0.85, cmap='viridis')
    ax.set_xlabel('Lagged Vol', fontsize=8)
    ax.set_ylabel('VIX', fontsize=8)
    ax.set_zlabel('Potential', fontsize=8)
    ax.view_init(elev=30, azim=240)
    ax.set_facecolor('white')
    for pane in [ax.xaxis, ax.yaxis, ax.zaxis]:
        pane.set_pane_color((1, 1, 1, 0))
    if row == 0:
        ax.set_title('Gated Combination', fontsize=11, fontweight='bold')
    gate_str = ', '.join(f'{g:.3f}' for g in gate_here)
    ax.text2D(0.02, 0.92, f'w=[{gate_str}]', transform=ax.transAxes, fontsize=8)

    # Row label
    axes[row, 0].text2D(-0.15, 0.5, regime_label, transform=axes[row, 0].transAxes,
                         fontsize=11, fontweight='bold', rotation=90, va='center')

plt.suptitle('Potential Surfaces by Market Regime (All Macro Features from Historical Buckets)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
from dieboldmariano import dm_test

actuals = realized_vols[-N:]
preds_model = model_vol_mean_preds[-N:]
preds_arch = np.array(arch_vol_preds[-N:])

loss_fn = lambda y, yhat: abs(y - yhat)
dm_stat, p_value = dm_test(actuals, preds_model, preds_arch, h=1, loss=loss_fn)
print(f'DM Statistic: {dm_stat}, p-value: {p_value}')

In [ ]:
absolute_errors_model = np.abs(actuals - model_vol_mean_preds)
absolute_errors_arch = np.abs(actuals - preds_arch)
print(f'Model MAE: {absolute_errors_model.mean()}, GARCH MAE: {absolute_errors_arch.mean()}')   

In [ ]:
from monte_carlo import monte_carlo_nested_returns
from calculate_var import compute_var_from_nested_returns

# Use slow features from first test observation for MC simulation
slow_x0 = scaled_test_df.iloc[0][slow_feature_names].values

vol_paths, nested_returns = monte_carlo_nested_returns(
    model=model,
    x0_scaled=scaled_test_df.iloc[0].values,
    n_steps=20,
    n_vol_paths=1000,
    n_return_paths_per_vol=100,
    vol_index=vol_index,
    vol_mean=vol_center,
    vol_std=vol_scale,
    dt=dt,
    slow_x=slow_x0,
)

# 1-day 95% VaR in return units
var_1d_95 = compute_var_from_nested_returns(
    nested_returns,
    alpha=0.05,
    horizon=1,
    return_type="simple",
    position_value=1.0
)

In [ ]:
from scipy.stats import norm
from calculate_var import one_step_model_var_from_nested_mc

alpha = 0.05
z_alpha = norm.ppf(alpha)

n_fast_features = len(features)

# Realized next-day returns aligned with test set
returns = data['returns'].values[train_split + 1 : train_split + 1 + len(test_X)]

# GARCH forecasts already built as arch_vol_preds
garch_sigma = np.asarray(arch_vol_preds, dtype=float)

# Align test states, returns, and garch
n = min(len(test_X), len(returns), len(garch_sigma))

X_test_aligned = np.asarray(test_X[:n])
returns = np.asarray(returns[:n])
garch_sigma = np.maximum(garch_sigma[:n], 1e-10)

# Pre-extract slow features for each test observation
slow_test_aligned = np.asarray(test_X[:n])[:, slow_indices]

# Analytic GARCH VaR under Gaussian assumption
garch_var_95 = z_alpha * garch_sigma   # negative number

# Model VaR via nested MC (1000 vol paths x 100 return paths = 100k samples per observation)
model_var_95 = []

for i in range(n):
    q_alpha = one_step_model_var_from_nested_mc(
        model=model,
        x0_scaled=X_test_aligned[i],
        n_vol_paths=1000,
        n_return_paths_per_vol=100,
        n_fast_features=n_fast_features,
        vol_index=vol_index,
        vol_mean=vol_center,
        vol_std=vol_scale,
        dt=dt,
        alpha=alpha,
        eps=1e-10,
        slow_x=slow_test_aligned[i],
    )
    model_var_95.append(q_alpha)

model_var_95 = np.asarray(model_var_95, dtype=float)
print(f"Model 1-step 95% VaR (return): {model_var_95.mean():.6f}")
print(f"GARCH 1-step 95% VaR (return): {garch_var_95.mean():.6f}")

model_exceed = returns < model_var_95
garch_exceed = returns < garch_var_95

model_exceed_rate = np.mean(model_exceed)
garch_exceed_rate = np.mean(garch_exceed)

print(f"Aligned observations: {n}")
print(f"Expected 95% VaR exceedance rate: {alpha:.4f}")
print(f"Model 95% VaR exceedance rate: {model_exceed_rate:.4f}")
print(f"GARCH 95% VaR exceedance rate: {garch_exceed_rate:.4f}")
print(f"Model exceedance count: {model_exceed.sum()} / {n}")
print(f"GARCH exceedance count: {garch_exceed.sum()} / {n}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -- ChatGPT made this visual to compare VaR exceedances of the surrogate model vs GARCH -- 

# ── VaR Exceedance Visualization: Model vs GARCH (same axes) ──
fig, ax = plt.subplots(figsize=(16, 6))

x = np.arange(n)

# Return path
ax.plot(x, returns, color='grey', alpha=0.4, lw=0.7, zorder=1, label='Realized Return')

# VaR lines
ax.plot(x, garch_var_95, color='#1f77b4', lw=1.2, alpha=0.9, zorder=2, label='GARCH(1,1) 95% VaR')
ax.plot(x, model_var_95, color='#ff7f0e', lw=1.2, alpha=0.9, zorder=2, label='Surrogate Model 95% VaR')

# Breach markers
garch_breach_idx = np.where(garch_exceed)[0]
model_breach_idx = np.where(model_exceed)[0]
ax.scatter(garch_breach_idx, returns[garch_breach_idx], color='#1f77b4', s=20, zorder=4,
           marker='v', edgecolors='navy', linewidths=0.5, label=f'GARCH breaches ({garch_exceed.sum()})')
ax.scatter(model_breach_idx, returns[model_breach_idx], color='#ff7f0e', s=20, zorder=4,
           marker='^', edgecolors='saddlebrown', linewidths=0.5, label=f'Model breaches ({model_exceed.sum()})')


# Annotation box
textstr = (
    f"  {'':22s} {'Model':}  {'GARCH':}\n"
    f"  {'Expected rate':22s} {alpha:.1%}    {alpha:.1%}\n"
    f"  {'Actual rate':22s} {model_exceed_rate:.1%}    {garch_exceed_rate:.1%}\n"
    f"  {'Breaches':18s} {model_exceed.sum():>7d}  {garch_exceed.sum():>7d}\n"
)
props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='grey', alpha=0.92)
ax.text(0.01, 0.02, textstr, transform=ax.transAxes, fontsize=8.5,
        verticalalignment='bottom', fontfamily='monospace', bbox=props)

ax.set_ylabel('Return', fontsize=10)
ax.set_xlabel('Test Observation', fontsize=10)
closer = 'Model' if abs(model_exceed_rate - alpha) < abs(garch_exceed_rate - alpha) else 'GARCH'
ax.set_title(
    f'95% VaR Backtesting \u2014 {closer} closer to expected {alpha:.0%} rate  '
    f'(Model: {model_exceed_rate:.2%}, GARCH: {garch_exceed_rate:.2%})',
    fontsize=12, fontweight='bold'
)
ax.legend(loc='upper right', fontsize=9)
ax.set_facecolor('#fafafa')

plt.tight_layout()
plt.show()


In [ ]:
# -- Here I also asked ChatGPT to use a t-test to analyze the residuals --

from scipy.stats import ttest_ind, ttest_1samp, f_oneway
import numpy as np

# ── VaR Calibration Residual Analysis ──
# Per-observation residual: breach_indicator[i] - alpha
# A perfectly calibrated model has E[residual] = 0.
model_resid = model_exceed.astype(float) - alpha
garch_resid = garch_exceed.astype(float) - alpha

# Ideal baseline: Bernoulli(alpha) — what a perfectly calibrated model looks like
rng = np.random.default_rng(42)
ideal_resid = rng.binomial(1, alpha, n).astype(float) - alpha

print(f"{'VaR Calibration Residual Analysis':=^65}")
print(f"  Residual[i] = breach_indicator[i] - {alpha}   (0 = perfect day)\n")
print(f"  {'':22s} {'Model':>10s} {'GARCH':>10s} {'Ideal':>10s}")
print(f"  {'─'*52}")
print(f"  {'Mean residual':22s} {model_resid.mean():>10.4f} {garch_resid.mean():>10.4f} {ideal_resid.mean():>10.4f}")
print(f"  {'Std':22s} {model_resid.std():>10.4f} {garch_resid.std():>10.4f} {ideal_resid.std():>10.4f}")
print()

# ── Test 1: One-sample t-test — is each model's mean residual = 0? ──
# H0: E[breach_indicator] = alpha  (model is well-calibrated)
model_t1, model_p1 = ttest_1samp(model_resid, 0.0)
garch_t1, garch_p1 = ttest_1samp(garch_resid, 0.0)

print(f"  1. One-sample t-test  (H\u2080: mean residual = 0, i.e. well-calibrated)")
print(f"  {'':22s} {'t':>8s} {'p':>10s}")
print(f"  {'─'*44}")
def verdict(p): return '\u2717 reject H\u2080' if p < 0.05 else '\u2713 fail to reject'
print(f"  {'Model':22s} {model_t1:>8.3f} {model_p1:>10.4f}  {verdict(model_p1)}")
print(f"  {'GARCH':22s} {garch_t1:>8.3f} {garch_p1:>10.4f}  {verdict(garch_p1)}")
print()